# QUEDA — Fiscalização de dados abertos do Crea-SP

Executa a coleta e a análise **a partir do Colab** (o tráfego sai por IPs do Google, não da sua máquina).

**Se o repositório for privado**: use `Arquivos → Upload` para subir um zip do repo e ajuste o caminho abaixo, ou gere um token de leitura no GitHub e use `https://TOKEN@github.com/...` na célula de clone (não salve o token no notebook).

> Aviso: todo achado é **indício**, não prova. Confirme na fonte antes de qualquer denúncia.

**Camada de IA (opcional)**: roda um modelo local via Ollama dentro do próprio Colab — sem chave de API e sem que nenhum dado do conselho saia da máquina. Pular a célula 2 não tira nenhum número do relatório.


In [ ]:
# 1) Obter o código
import os, subprocess
REPO = 'github.com/caiofvserra/queda.git'
BRANCH = 'claude/crea-sp-audit-tool-m9nw7t'
if not os.path.exists('queda'):
    r = subprocess.run(['git','clone','--depth','1','-b',BRANCH,f'https://{REPO}'])
    if r.returncode != 0:  # repositório privado -> pedir token sem gravá-lo no notebook
        from getpass import getpass
        tok = getpass('Repo privado. Cole um token GitHub fine-grained (Contents: read-only): ')
        subprocess.run(['git','clone','--depth','1','-b',BRANCH,f'https://{tok}@{REPO}'], check=True)
        del tok
%cd queda
!pip -q install -r requirements.txt
!pip -q install networkx   # intermediação exata no grafo da rede


In [ ]:
# 2) OPCIONAL — camada de IA rodando LOCAL (Ollama)
# Nenhum número do relatório depende disto: todo cálculo é determinístico.
# O modelo entra só onde regra nenhuma serve — enquadrar o resíduo da
# taxonomia, ler a prosa das atas e redigir o resumo executivo.
#
# Roda 100% na máquina do Colab: nenhum dado do conselho sai daqui, e não
# há chave nem custo por token. O download é de ~4,7 GB, uma vez por sessão.
# Para pular, basta NÃO executar esta célula.
#
# Ative a GPU antes: Ambiente de execução -> Alterar tipo -> GPU (T4).
MODELO = 'qwen2.5:7b-instruct'   # leve: 'llama3.2:3b' | melhor: 'gemma2:9b'

import os, subprocess, time
os.environ['QUEDA_LLM_MODELO'] = MODELO

!curl -fsSL https://ollama.com/install.sh | sh
subprocess.Popen(['ollama', 'serve'],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull {MODELO}
!python main.py llm


In [ ]:
# 3) Coleta + análise + relatório
# --desde auto: sonda anos retroativamente até a API esvaziar (meta: pré-2016)
# Primeira execução completa pode demorar (throttle de ~1 req/s por educação com as APIs).
!python main.py all --desde auto --ate 2026


In [ ]:
# 4) Ver o relatório aqui mesmo
from IPython.display import HTML, display
display(HTML(open('output/report.html').read()))


In [ ]:
# 5) Baixar o relatório e os CSVs de evidência
!zip -qr resultado_queda.zip output/
from google.colab import files
files.download('resultado_queda.zip')


## Reexecuções
O cache em `data/raw/` evita repetir chamadas — para atualizar um período, apague os arquivos correspondentes ou a pasta `data/` inteira.

**Se a coleta parar com aviso de rate-limit (403/429)**: rode a MESMA célula de novo. O que já veio está em cache e a execução retoma de onde parou; nada é recoletado.

Análises sem nova coleta: `!python main.py analyze` (aceita `--sem-llm`).

A aba **Diagnóstico da coleta** do relatório diz o que veio, o que faltou e qual análise ficou desligada por falta de qual fonte — comece por ela.
